# Buddy — custom wake-word training

Trains an **openWakeWord** model for "hey buddy" and its variants from
**synthetic speech** — no recordings needed, and nothing leaves the machine at
inference: the output is a ~1MB ONNX running locally at a few % CPU.

**Runtime: GPU** (Runtime → Change runtime type → T4). Budget ~1–2 hours.

### Which phrases wake it

Carrier forms only — `hey buddy`, `hi buddy`, `hello buddy`, `okay buddy`.
Three syllables with crisp /b/ /d/ plosives is the detection sweet spot, and
several carriers make it robust to however you say it.

**Bare `buddy` is a `custom_negative_phrase`, deliberately.** It isn't merely
left out: a 2-syllable bare word would subsume the carriers — once the model
fires on "buddy" alone, "hey" stops mattering and every conversational "buddy"
is a wake event. Listing it as a negative teaches that the carrier is *required*.

### How this runs

`openwakeword/train.py` drives everything from one YAML config — it generates
the clips, augments them, computes features and trains, in three commands. An
earlier version of this notebook hand-rolled those steps and called `auto_train`
directly; that is an internal API taking feature arrays, not directories.

## 1. GPU check

In [ ]:
import torch
assert torch.cuda.is_available(), "NO GPU — Runtime → Change runtime type → T4, then re-run"
print(torch.cuda.get_device_name(0))

## 2. Install

Version pins here are load-bearing, each one learned from a failure:
- **piper-sample-generator @ v2.0.0** — master was restructured into a module with
  no `generate_samples.py`, and the PyPI package expects a `piper_train` that
  only this tag vendors.
- **piper-phonemize-cross** — the original ships no wheel for Colab's Python 3.12.
- **openWakeWord from source** — the PyPI wheel is inference-only, no `train` module.

In [ ]:
import os
os.chdir("/content")

!git clone -q https://github.com/dscripka/openwakeword /content/openwakeword
!pip install -q -e /content/openwakeword

!git clone -q --branch v2.0.0 --depth 1 \
  https://github.com/rhasspy/piper-sample-generator /content/piper-sample-generator
!mkdir -p /content/piper-sample-generator/models
!wget -q -O /content/piper-sample-generator/models/en_US-libritts_r-medium.pt \
  https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt

!pip install -q piper-phonemize-cross webrtcvad
!pip install -q torchinfo torchmetrics mutagen speechbrain audiomentations \
                torch-audiomentations acoustics pronouncing datasets \
                deep-phonemizer onnx scipy tqdm pyyaml

# PyTorch 2.6 flipped torch.load's weights_only default to True, which the
# 2024-era generator can't satisfy (its checkpoint holds a SynthesizerTrn).
# Safe here: official release asset, over HTTPS, on a disposable VM.
import re, pathlib
_p = pathlib.Path("/content/piper-sample-generator/generate_samples.py")
_src = _p.read_text()
_new = re.sub(r"torch\.load\(([^)]*)\)",
              lambda m: m.group(0) if "weights_only" in m.group(1)
                        else f"torch.load({m.group(1)}, weights_only=False)", _src)
if _new != _src:
    _p.write_text(_new); print("patched torch.load for PyTorch 2.6+")

from piper_phonemize import phonemize_espeak      # fails fast if the fork is wrong
print("setup OK")

## 3. Training data

Background audio and room impulse responses — this is what teaches the model
that ordinary sound is *not* the wake word. Several GB; the slowest cell.

In [ ]:
import os, numpy as np, scipy.io.wavfile, datasets
from pathlib import Path
from tqdm import tqdm
os.chdir("/content")

# room impulse responses — so it survives being spoken to across a desk
os.makedirs("mit_rirs", exist_ok=True)
if not os.listdir("mit_rirs"):
    rirs = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses",
                                 split="train", streaming=True)
    for row in tqdm(rirs, desc="RIRs"):
        scipy.io.wavfile.write(f"mit_rirs/{row['audio']['path'].split('/')[-1]}",
                               16000, (row['audio']['array'] * 32767).astype(np.int16))

# AudioSet — general background noise
if not os.path.exists("audioset_16k") or not os.listdir("audioset_16k"):
    os.makedirs("audioset", exist_ok=True)
    !wget -q -O audioset/bal_train09.tar https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar
    !cd audioset && tar -xf bal_train09.tar
    os.makedirs("audioset_16k", exist_ok=True)
    ds = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
    ds = ds.cast_column("audio", datasets.Audio(sampling_rate=16000))
    for row in tqdm(ds, desc="audioset"):
        name = row["audio"]["path"].split("/")[-1].replace(".flac", ".wav")
        scipy.io.wavfile.write(f"audioset_16k/{name}", 16000,
                               (row["audio"]["array"] * 32767).astype(np.int16))

# FMA — music, a common false-positive source
if not os.path.exists("fma") or not os.listdir("fma"):
    os.makedirs("fma", exist_ok=True)
    fma = iter(datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
               .cast_column("audio", datasets.Audio(sampling_rate=16000)))
    for _ in tqdm(range(1 * 3600 // 30), desc="fma"):
        row = next(fma)
        name = row["audio"]["path"].split("/")[-1].replace(".mp3", ".wav")
        scipy.io.wavfile.write(f"fma/{name}", 16000,
                               (row["audio"]["array"] * 32767).astype(np.int16))

# pre-computed negative features + the false-positive validation set
if not os.path.exists("openwakeword_features_ACAV100M_2000_hrs_16bit.npy"):
    !wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
if not os.path.exists("validation_set_features.npy"):
    !wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

for p in ["mit_rirs", "audioset_16k", "fma"]:
    print(f"{p}: {len(os.listdir(p))} files")

## 4. The config

In [ ]:
import yaml, os
os.chdir("/content")

config = {
    "model_name": "hey_buddy",
    "target_phrase": ["hey buddy", "hi buddy", "hello buddy", "okay buddy"],

    # bare "buddy" leads this list on purpose — see the note at the top
    "custom_negative_phrases": [
        "buddy", "buddies", "body", "study", "muddy", "buttery", "bloody",
        "budding", "butter", "bunny", "buggy", "somebody", "everybody",
        "already", "hey buzz", "hey bunny", "hi body", "hey there",
        "my buddy", "your buddy", "the buddy system",
    ],

    "n_samples": 20000,          # the documented minimum; raise for a better model
    "n_samples_val": 2000,
    "steps": 20000,
    "target_accuracy": 0.6,
    "target_recall": 0.25,
    "target_false_positives_per_hour": 0.2,
    "max_negative_weight": 1000,

    "model_type": "dnn",
    "layer_size": 32,
    "tts_batch_size": 50,
    "augmentation_batch_size": 16,
    "augmentation_rounds": 1,
    "batch_n_per_class": {"ACAV100M_sample": 1024, "adversarial_negative": 50, "positive": 50},

    "piper_sample_generator_path": "/content/piper-sample-generator",
    "output_dir": "/content/hey_buddy_model",
    "rir_paths": ["/content/mit_rirs"],
    "background_paths": ["/content/audioset_16k", "/content/fma"],
    "background_paths_duplication_rate": [1, 1],
    "false_positive_validation_data_path": "/content/validation_set_features.npy",
    "feature_data_files": {
        "ACAV100M_sample": "/content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy"},
}

os.makedirs(config["output_dir"], exist_ok=True)
with open("/content/hey_buddy.yaml", "w") as f:
    yaml.dump(config, f, sort_keys=False)
print(yaml.dump(config, sort_keys=False))

## 5. Train

Three phases. Run them in order — each depends on the last. Generation is the
long one (~20k clips through Piper).

In [ ]:
!cd /content && python openwakeword/openwakeword/train.py \
    --training_config /content/hey_buddy.yaml --generate_clips

In [ ]:
!cd /content && python openwakeword/openwakeword/train.py \
    --training_config /content/hey_buddy.yaml --augment_clips

In [ ]:
!cd /content && python openwakeword/openwakeword/train.py \
    --training_config /content/hey_buddy.yaml --train_model

## 6. Validate

A model scoring 0.99 on the phrase is useless if it also scores 0.8 on "study".
The **gap** is what matters, and the app's threshold goes inside it.

In [ ]:
import numpy as np, glob, scipy.io.wavfile, os
from openwakeword.model import Model as OWWModel

onnx = glob.glob("/content/hey_buddy_model/**/*.onnx", recursive=True)
assert onnx, "no ONNX produced — check the train_model step's output"
MODEL_PATH = onnx[0]
print("model:", MODEL_PATH)

oww = OWWModel(wakeword_models=[MODEL_PATH], inference_framework="onnx")
key = list(oww.models.keys())[0]

def score(pattern, limit=150):
    out = []
    for path in glob.glob(pattern, recursive=True)[:limit]:
        rate, audio = scipy.io.wavfile.read(path)
        oww.reset()
        peaks = [oww.predict(audio[i:i+1280])[key]
                 for i in range(0, len(audio) - 1280, 1280)]
        if peaks:
            out.append(max(peaks))
    return np.array(out)

pos = score("/content/hey_buddy_model/positive*/**/*.wav")
neg = score("/content/hey_buddy_model/negative*/**/*.wav")
if len(pos) and len(neg):
    p05, n95 = np.percentile(pos, 5), np.percentile(neg, 95)
    print(f"positives  mean {pos.mean():.3f}  p05 {p05:.3f}")
    print(f"negatives  mean {neg.mean():.3f}  p95 {n95:.3f}")
    print(f"\nseparation {p05 - n95:+.3f}  "
          f"({'usable' if p05 - n95 > 0.2 else 'TOO TIGHT — more steps or negatives'})")
    print(f"suggested threshold: {(p05 + n95) / 2:.2f}")
else:
    print("no clips found to score — check output_dir layout")

## 7. Export

Download the ONNX and drop it in `src/voice/hey_buddy.onnx`, then wire it at the
construction site in `src/menubar/app.py` (~line 285):

```python
from src.paths import resource_path

self._wake = WakeWordListener(
    on_wake=...,
    gate=...,
    model_name=str(resource_path("src", "voice", "hey_buddy.onnx")),
    threshold=0.5,   # the suggested threshold from step 6
)
```

Then set `"wake_word": true` under `features` in `config.json` and rebuild.

Tuning: raise the threshold if it fires during ordinary conversation, lower it
if it misses you. The listener already gates itself while Buddy is speaking,
mid-turn, or while push-to-talk holds the mic, so you are only tuning against
background speech.

In [ ]:
from google.colab import files
files.download(MODEL_PATH)